## 1、流式调用 、非流式调用

In [1]:
# 非流式调用
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
import dotenv
from openai import max_retries
from sympy.physics.units import temperature

dotenv.load_dotenv()
# 1、获取大模型的实例
model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    temperature = 0.5,
)
res = model.invoke("你好，你是谁")
print(res)

C:\Users\m1881\miniconda3\envs\LangChainProj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我可以帮你处理上传的各种文件，比如图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我拥有128K的上下文长度，完全免费使用，还支持联网搜索功能（需要你在Web/App中手动开启）。\n\n我很乐意为你提供各种帮助，无论是回答问题、协助思考问题、创作内容，还是进行对话交流。你可以通过官方应用商店下载我的App来使用。\n\n有什么我可以帮你的吗？我会尽我所能为你提供热情、细腻的服务！✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 7, 'total_tokens': 148, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '08c17972-b9af-4b3b-a43a-8591723d06aa', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--86243e29-5c95-4df3-b0a6-8daa61bf653c-0' usage_metadata={'input_tokens': 7, 'output_tokens': 141, 'total_tokens': 148, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [10]:
# 流式调用，通过model.stream方法去调用，
# 返回的是一个生成器，通过迭代生成器的方式，得到结果
res = model.stream("你好，你是谁")

In [11]:
for chunk in res:
    print(chunk.content,end="")

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动开启）。

我很乐意为你提供各种帮助，无论是回答问题、协助工作、学习辅导，还是日常聊天，我都会热情细致地为你服务！有什么我可以帮你的吗？✨

## 2、批次调用、非批次调用

In [12]:
# 批次调用，通过model.batch方法实现，底层原理就是通过多线程的方式去调用，
#
messages = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
res = model.batch(messages)

## 3、同步调用、异步调用

In [15]:
# 同步调用：多次请求之间串行处理，B请求需要A请求完成之后，再发出请求，得到响应
messagess = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
import time
start_time = time.time()
res = [model.invoke(messages) for messages in messagess]
end_time = time.time()
print(f"总耗时:{end_time - start_time}")

总耗时:17.451826810836792


In [19]:
# 异步调用：model.ainvoke方法，返回一个协程对象，把多个协程对象可以打包成一个协程对象，
# await最终的协程对象，就能够实现异步调用
# 异步调用，能够提高程序的性能。相对于batch调用而言，能够减少资源（线程数）使用量
import asyncio
async def gather_task(messages:list):
    # 调用ainvoke并不会真正地发起请求
    tasks = [model.ainvoke(message_list) for message_list in messages]
    return await asyncio.gather(*tasks)
gather_task(messagess)

<coroutine object gather_task at 0x00000176C1CA74C0>

In [20]:
await gather_task(messagess)

[AIMessage(content='《春日来信》\n\n融雪在屋檐写下第一行诗\n柳枝蘸着湖水练习签名\n杏花拆开三月的信封时\n整片天空都飘满押韵的芬芳\n\n蚯蚓在泥土里修订平仄\n燕子带着平仄飞过山岗\n蒲公英撑开绒毛伞之前\n春天已把韵脚藏进每粒星光\n\n当犁铧翻开大地的诗稿\n所有种子开始酝酿意象\n待到布谷鸟开始朗诵时\n每片叶子都举着绿色的印章', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 12, 'total_tokens': 122, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'c63dcbff-5349-40cd-af54-255898a99b1a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--d2ccd8e4-cc94-44b9-be03-6b71ea830a90-0', usage_metadata={'input_tokens': 12, 'output_tokens': 110, 'total_tokens': 122, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 AIMessage(content='《夏日的密语》\n\n蝉鸣织成透明的网\n筛落满地碎银\n荷叶托起整个正午的摇晃\n\n蜻蜓在湖面写下断章\n风一吹就散了\n只剩下水纹慢慢生长